# Hindi–Sanskrit SLM: data, model, and training

This notebook is the map of experiment `010-hindi-sanskrit-slm`. It follows the build in three parts:

1. **Data sourcing and preparation** — which Hindi and Sanskrit corpora are kept, and how a line is cleaned.
2. **Model architecture** — the 12,000-piece tokenizer, the tied embedding, and the 10,942,720-parameter decoder, including the KV cache.
3. **Training pipeline** — base pretraining, supervised fine-tuning, and preference alignment.

Run the code cells from this directory, or from the repo root. They only read configs, a few JSONL rows, and logs. They do not start a training job.

Two of the three training stages exist in code. Preference alignment (DPO) is the planned third stage and is not implemented. The notebook says so wherever that matters.


## How the three stages fit together

```text
raw corpora
    -> clean, filter, dedupe, hold out benchmarks
    -> SentencePiece Unigram (12,000 pieces)
         |
         +--> stage 1  base pretraining
         |       monolingual Hindi and Sanskrit
         |       control tokens <hi> and <sa>
         |       stop at 100M tokens, then optional 200M, 300M, ...
         |
         +--> stage 2  supervised fine-tuning
         |       parallel Hindi–Sanskrit pairs
         |       fresh AdamW, weights loaded from one pretrain checkpoint
         |       translation, Sanskritized Hindi, correction
         |
         +--> stage 3  preference alignment   [not built]
                 chosen vs rejected answers
                 DPO on top of one fine-tuned checkpoint
```

Stage 1 teaches the model to continue Devanagari text. Stage 2 teaches it to follow a task tag. Stage 3 would teach it to prefer one answer over another. A fine-tune is never used as the start of another fine-tune, and a later pretrain boundary does not resume a fine-tune.


## 1. Data sourcing and preparation

### 1.1 Corpora, marked with date and quality

| Dataset | Snapshot / prepared | Quality | Licence and restriction | In training |
| --- | --- | --- | --- | --- |
| Wikipedia `20231101.hi` | Snapshot 2023-11-01, prepared 2026-09-25 | silver | CC-BY-SA-4.0, share-alike | Yes |
| Wikipedia `20231101.sa` | Snapshot 2023-11-01, prepared 2026-09-25 | silver | CC-BY-SA-4.0, share-alike | Yes |
| Samasāmayik train | Prepared 2026-09-25 | gold | Research only. No commercial use, no redistribution | Yes |
| BPCC Hindi–Sanskrit pivot | Prepared 2026-09-25 | silver | CC0 | Yes |
| IN22 | Prepared 2026-09-25 | silver | CC-BY-SA-4.0 | Yes, added as silver pairs |
| FLORES-200 | Prepared 2026-09-25 | silver | CC-BY-SA-4.0 | Yes, added as silver pairs |
| Samasāmayik official test | Prepared 2026-09-25 | gold | Research only, and it is the held-out test | No |
| Sangraha | — | unreviewed | Licence unclear | No |

Quality `gold` means a curated bitext from the Samasāmayik release. Quality `silver` means Wikipedia text or a pivoted / benchmark bitext. The prepared date is the day this experiment froze the files. The Wikipedia snapshot date is the dump, which is older than the download.

The training rule is: if the licence allows training, the rows are used, and the licence's own limits stay attached to the rows. Research-only stays research-only. An unclear licence stays out. The official Samasāmayik test stays out because that split is the held-out restriction, not because the licence is missing.

Sanskritized Hindi is a register, not the default. Everyday Hindi with loanwords is valid text. A preference method that always punished loanwords was rejected for this version, which is why stage 3 is not a loanword penalty.


### 1.2 What was selected, and what was dropped

Preparation lives in `src/prep/`. The order is: download, hash the held-out lines, align pairs, clean, drop duplicates, drop leaks, assign a validation split, write JSONL.

A monolingual paragraph is kept only when all of these hold:

- Unicode NFC, zero-width characters removed, whitespace collapsed
- length between 20 and 4,000 characters
- not repetitive (if it has at least 8 words, unique words must be at least 30% of the words)
- at least 90% of its letters are Devanagari (`MONO_SCRIPT_MIN = 0.9`)
- a function-word check does not label it as the other language
- its exact text hash is not in the banned benchmark set

A parallel pair is kept only when both sides pass the same length and repetition checks, both sides are at least 60% Devanagari, and the character-length ratio is between 0.3 and 3. Each kept pair is stored twice: Hindi to Sanskrit, and Sanskrit to Hindi.

Exact duplicates are removed with a SHA-256 of the text (or of task + source + target). The validation split is 1% of rows, chosen from the first bytes of that hash, so the split is stable if the file is rebuilt. Fine-tuning reads `split == train` only. Evaluation reads `split == val`.


In [ ]:
from pathlib import Path
import json
import yaml

HERE = Path.cwd()
ROOT = HERE if (HERE / "configs" / "model_10m.yaml").exists() else HERE / "experiments" / "language" / "010-hindi-sanskrit-slm"
print("experiment", ROOT)

report = json.loads((ROOT / "data/manifests/prepare_report.json").read_text())
for key in (
    "pretrain_rows", "pretrain_hi", "pretrain_sa",
    "sft_rows", "sft_train", "sft_val",
    "gold_rejected", "silver_rejected",
    "parallel_dupes", "parallel_leaks", "mono_dupes", "mono_leaks",
    "benchmark_hashes",
):
    print(f"{key:22} {report.get(key, 0):,}")

def first_matching(path, predicate):
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            if predicate(row):
                return row

pretrain = ROOT / "data/clean/pretrain.jsonl"
sft = ROOT / "data/clean/sft.jsonl"
hi = first_matching(pretrain, lambda row: row["language"] == "hi")
sa = first_matching(pretrain, lambda row: row["language"] == "sa")
val = first_matching(sft, lambda row: row["task"] == "translate_hi_sa" and row.get("split") == "val")
print("\npretrain file bytes", pretrain.stat().st_size)
print("sft file bytes     ", sft.stat().st_size)
print("Hindi source/licence", hi["source"], hi["license"], "script", hi["script_ratio"])
print("Sanskrit source/licence", sa["source"], sa["license"], "script", sa["script_ratio"])
print("val task", val["task"], "quality", val["quality"], "dataset", val["dataset"])
print("source", val["source"][:160])
print("target", val["target"][:160])


The numbers above are the cleaned corpus, not the token budget. Pretraining repeats these paragraphs until 100 million target tokens. The supervised file is the parallel data for stage 2. About 12,000 gold lines and a handful of silver lines failed the pair checks. About 41,000 Wikipedia paragraphs were exact duplicates. 7,392 benchmark hashes are banned. Zero monolingual leaks means no Wikipedia paragraph matched a held-out test line.

Row types:

- Pretrain: `language`, `text`, `source`, `license`, `script_ratio`, `split`
- Supervised translation: `task`, `source`, `target`, `dataset`, `quality` (`gold` or `silver`), `license`, `split`


## 2. Model architecture (about 10M parameters)

### 2.1 Tokenizer

The vocabulary is the embedding table, so a large multilingual tokenizer would spend the whole budget on embeddings. The tokenizer is a SentencePiece Unigram trained on an equal character budget of the cleaned Hindi and Sanskrit paragraphs (about 20.4 million characters each).

Training flags: character coverage 1.0, identity normalization (nukta, candrabindu, visarga, and danda are not rewritten), no byte fallback, no dummy prefix. Reserved pieces are `<unk> <pad> <bos> <eos>` plus the ten control tags.

Three sizes were trained and scored on held-out paragraphs. The score is tokens per word, unknown-token rate, and whether nukta, danda, and a long compound survive a round trip.


In [ ]:
compare = json.loads((ROOT / "tokenizers/compare.json").read_text())
print("train characters", {lang: f"{n:,}" for lang, n in compare["train_chars"].items()})
print(f"{'vocab':8} {'hi tok/word':12} {'sa tok/word':12} {'unk hi':12} {'round-trip'}")
for size, stats in compare["sizes"].items():
    print(
        f"{size:8} {stats['hi']['tokens_per_word']:12.3f} "
        f"{stats['sa']['tokens_per_word']:12.3f} "
        f"{stats['hi']['unk_rate']:12.2e} {str(stats['roundtrip']):>10}"
    )
print("chosen", compare["chosen"])
print(compare["reason"])


12,000 pieces wins. Hindi is about 1.47 tokens per word and Sanskrit about 2.27. Unknown tokens are effectively zero, and the Devanagari probes round-trip. Sanskrit is still more fragmented than Hindi, which is the reason to spend embedding rows on vocabulary instead of staying at 8,000.

The chosen file is `tokenizers/sp.model`. Arabic and Latin pieces are not added. Loanwords that are already written in Devanagari stay in the text.


### 2.2 Embeddings and the blueprint

`configs/model_10m.yaml` is the blueprint. The input embedding and the output head are the same matrix (`tie_embeddings: true`), so those 3,072,000 weights are counted once.

| Piece | Shape | Count |
| --- | --- | --- |
| Token embedding, tied to the output head | 12,000 × 256 | 3,072,000 |
| Query, key, value, output projection in one block | 8 and 2 heads, head dim 32 | 163,840 |
| SwiGLU (three matrices) | 256 × 640 | 491,520 |
| Two RMSNorm scales in the block | 256 each | 512 |
| Twelve blocks | | 7,870,464 |
| Final RMSNorm | 256 | 256 |
| **Trainable total** | | **10,942,720** |

Rotary tables are buffers, not parameters. Base is 100,000. The pair rotation matches experiment 008. Context length is 1,024. Dropout is 0.05.

Grouped-query attention stores keys and values for 2 heads and repeats them to match 8 query heads. That is the memory saving at inference time. SwiGLU is the feed-forward. Each block is pre-norm: RMSNorm, then attention or SwiGLU, then a residual add. The output projections start smaller than the other matrices (`0.02 / sqrt(2 * n_layer)`) so the residual path does not explode at step 0.


In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.model import count_parameters

config = yaml.safe_load((ROOT / "configs/model_10m.yaml").read_text())
pretrain_cfg = yaml.safe_load((ROOT / "configs/pretrain.yaml").read_text())
sft_cfg = yaml.safe_load((ROOT / "configs/sft.yaml").read_text())

vocab, width = config["vocab_size"], config["n_embd"]
query = config["n_head"] * config["head_dim"]
key_value = config["n_kv_head"] * config["head_dim"]
embed = vocab * width
attention = width * query + 2 * width * key_value + query * width
feed_forward = 3 * width * config["intermediate"]
block = attention + feed_forward + 2 * width

print("type                 dense decoder, tied embedding, GQA, SwiGLU, RoPE")
print(f"parameters           {count_parameters(config):,}")
print(f"vocab / width        {vocab:,} / {width}")
print(f"layers               {config['n_layer']}")
print(f"heads                {config['n_head']} query, {config['n_kv_head']} kv, dim {config['head_dim']}")
print(f"embedding            {embed:,}")
print(f"one block            {block:,}")
print(f"context / dropout    {config['block_size']} / {config['dropout']}")
print(f"rope base            {config['rope_theta']:,}")
print(f"pretrain lr          {pretrain_cfg['peak_lr']} -> {pretrain_cfg['min_lr']}")
print(f"sft lr               {sft_cfg['peak_lr']} -> {sft_cfg['min_lr']}")


### 2.3 KV cache and the other implementation choices

Generation is a prefill of the prompt, then one new token at a time. `KVCache` holds keys and values for every layer, shaped `(layers, batch, context, kv_heads, head_dim)`. Only the 2 key-value heads are stored, not the 8 query heads. The next token attends to that cache instead of recomputing the whole prefix. `src.smoke` checks that a cached prefill and a cached decode match a full forward within 1e-4.

Other choices that are in the code, and why:

- **Tied embedding.** Saves 3,072,000 parameters versus an untied head, which is what keeps the model near 10M at vocab 12,000.
- **GQA rather than multi-head attention.** Same query width, a quarter of the KV memory.
- **SwiGLU.** Three linear maps instead of two. The intermediate width 640 was chosen so the parameter total lands on 10,942,720.
- **AdamW on matrices only.** Weight decay 0.1 applies to rank-2 weights. Norm scales are not decayed. Betas are 0.9 and 0.95. Gradients are clipped at 1.0.
- **Cosine schedule.** Linear warmup for 2% of the token budget, then cosine down to 10% of the peak. For the fine-tune, the schedule length is fit to the real supervised tokens per step (short batches), not to a full context of 1,024.
- **Non-finite steps are skipped.** A single NaN loss or gradient on MPS does not update the weights. Three in a row stop the run. The supervised-token count is read on the CPU so a bad device scalar cannot pretend the budget is finished.
- **Checkpoints.** `runs/pretrain/latest` every 500 steps, plus `tok-100m`, `tok-200m`, and so on at each 100M-token boundary. A checkpoint stores the model, optimizer, schedule, data hash, tokenizer bytes, and RNG state.

What was considered and left out of this 10M model: mixture-of-experts, multi-head latent attention, and consistency-entropy decoding. Those are other experiments in this repo. They do not change this blueprint.


## 3. Training pipeline

### Stage 1 — base pretraining

```text
Wikipedia paragraphs
    -> normalize, script filter, dedupe, leakage filter
    -> data/clean/pretrain.jsonl
    -> Unigram encode as <hi>text<eos> or <sa>text<eos>
    -> pack each language into blocks of 1,024
    -> next-token cross-entropy
```

The label at each position is the token itself. The model shifts by one inside the loss, so position `t` predicts token `t+1`. `tokens_seen` counts those target positions.

Optimizer: AdamW, peak learning rate 6e-4, floor 6e-5, batch 4, context 1,024. The first stop is 100 million tokens. That checkpoint is `runs/pretrain/tok-100m` (step 24,438, 100,000,296 tokens, final loss 4.81). Later stops add 100 million tokens each and resume the same optimizer. Warmup does not restart on resume.

From that checkpoint, Hindi continuations are still short fragments. Sanskrit continuations are short clauses. That is expected before stage 2: the model has only been asked to continue text.


### Stage 2 — supervised fine-tuning

```text
parallel Hindi–Sanskrit pairs (train split only)
    -> <task><src>source<tgt>target<eos>
    -> labels are -100 through <tgt>
    -> cross-entropy on the answer only
    -> new AdamW, weights copied from one pretrain checkpoint
```

Tasks in the current file are `translate_hi_sa` and `translate_sa_hi`. The control tags `<sanskritized_hi>`, `<standard_hi>`, `<correct_hi>`, and `<correct_sa>` are reserved in the tokenizer and tested on the fixture. They are not a large part of `data/clean/sft.jsonl` yet.

The fine-tune budget is 20 million supervised tokens. Peak learning rate is 2e-4, floor 2e-5. The pretrain optimizer is discarded. Validation rows (3,227) are not in this loop. The checkpoint name is `runs/sft/from-tok-100m`.

A supervised batch is short, on the order of 90 target tokens, because rows are padded to the longest row in the batch rather than to 1,024. The cosine length uses that real width. Otherwise the learning rate would fall to the floor after a few percent of the budget.


In [ ]:
from src.tokenizer import Tokenizer
from src.data import encode_sft, supervised_count

tokenizer = Tokenizer(ROOT / "tokenizers/sp.model")
row = {
    "task": "translate_hi_sa",
    "source": "वह पुस्तक पढ़ता है।",
    "target": "सः पुस्तकं पठति।",
}
ids, labels = encode_sft(row, tokenizer)
tgt_at = ids.index(tokenizer.id_of("<tgt>"))
print("pieces", tokenizer.vocab_size, "pad", tokenizer.pad_id, "eos", tokenizer.eos_id)
print("sequence length", len(ids), "supervised", supervised_count(labels))
print("prompt ignored:", labels[: tgt_at + 1] == [-100] * (tgt_at + 1))
print("answer:", tokenizer.decode([i for i in labels if i != -100]))


### Stage 3 — preference alignment

This stage is the planned last step. It is not in the repository.

```text
preference pairs (chosen answer, rejected answer) for the same prompt
    -> DPO against the stage-2 model as the reference
    -> one aligned checkpoint
```

Direct preference optimization would raise the probability of the chosen answer and lower the probability of the rejected one, without training a separate reward model. The pairs are not collected yet. A useful pair compares two answers to the same source: a meaning-preserving translation against a fluent but wrong one, or a Sanskritized rewrite against a rewrite that changes the meaning.

What this stage will not be: a penalty on every Perso-Arabic or English loanword. That would treat ordinary Hindi as an error. Semantic correctness stays ahead of etymological purity.

Until those pairs exist, the model that can be shipped from this experiment is the stage-2 checkpoint, scored on the validation split.


### Where the runs stand

The cells below print the tail of the training logs if they are on disk. Pretraining to 100M is finished. The fine-tune from that checkpoint is a separate process. The 200M pretrain boundary has not been started.


In [ ]:
def tail(relative, n=4):
    path = ROOT / relative
    print(f"\n# {relative}")
    if not path.exists():
        print("not written yet")
        return
    for line in path.read_text(encoding="utf-8").splitlines()[-n:]:
        print(line)

tail("runs/pretrain/train.log")
tail("runs/sft/train.log")
print("\ntok-100m", (ROOT / "runs/pretrain/tok-100m/checkpoint.pt").exists())
print("sft final", (ROOT / "runs/sft/from-tok-100m/checkpoint.pt").exists())


## Tests and validation

Automated checks, all under `tests/`:

| File | What it locks |
| --- | --- |
| `test_prep.py` | Nukta survives normalization, short lines drop, Hindi vs Sanskrit labels, exact dedupe, leakage, both translation directions |
| `test_tokenizer.py` | Nukta and danda round-trip; the vocab does not grow Arabic or Latin pieces |
| `test_model.py` | Parameter count is 10,942,720, embedding and head share storage, query/key shapes, causal mask, finite backward |
| `test_data.py` | Fixture rows are non-empty, loss is masked through `<tgt>`, `<standard_hi>` is fully supervised |
| `test_train.py` | Checkpoint names, a resume across two token boundaries, a fresh SFT optimizer, an overfit whose loss falls |

`src.smoke` is the hardware check: the full model on CUDA, MPS, or CPU, cache gap under 1e-4, one AdamW step with a finite loss.

After stage 2, `src.evaluate` scores the validation split only: mean cross-entropy, chrF of a copy baseline (repeat the source), and chrF of the greedy generation after `<tgt>`. The official Samasāmayik test stays out of that number. IN22 and FLORES are now silver training data, so a score on those files is no longer a clean held-out result. Published full-size models sit near chrF++ 50 on the in-domain test and drop into the 30s on IN22. Those figures are a comparison point for a later evaluation, not a pass mark for this 10M model.


## Capability and shortcomings

What the 100M pretrained checkpoint can do:

- continue Sanskrit well enough to form a clause
- continue Hindi only as a short, often broken fragment
- store and resume a training run, including the optimizer

What it cannot do yet:

- follow `<translate_hi_sa>`, `<translate_sa_hi>`, or `<sanskritized_hi>` — those tags are unused until stage 2 finishes
- match a 1B translation model on chrF
- prefer a meaning-preserving answer over a fluent wrong one — that is stage 3, and the pairs do not exist

Structural limits that will remain after stage 2:

- about 7 million pretraining words, repeated, not a full web corpus
- Sanskrit costs more tokens per word than Hindi, so the same context holds less Sanskrit
- the language-id filter is a function-word heuristic, not a measured 98% classifier
- Samasāmayik is research-only, so this training mix cannot be redistributed or used commercially
- MPS training occasionally emits a non-finite loss; those steps are skipped, but they are a sign the run is numerically tight


## Commands

From `experiments/language/010-hindi-sanskrit-slm`:

```bash
uv sync
uv run python -m src.smoke
uv run pytest -q
uv run python scripts/prepare_data.py --hi-words 5000000 --sa-words 5000000
uv run python scripts/compare_tokenizers.py
uv run python -m src.train --stage pretrain --until-tokens 100000000
uv run python -m src.train --stage sft --init runs/pretrain/tok-100m
uv run python -m src.evaluate --checkpoint runs/sft/from-tok-100m --data data/clean/sft.jsonl
```

The 200M pretrain boundary, only when you ask for it:

```bash
uv run python -m src.train --stage pretrain --resume runs/pretrain/tok-100m --until-tokens 200000000
```


## Summary

Data comes from Hindi and Sanskrit Wikipedia plus Samasāmayik and BPCC pairs, after NFC cleanup, a Devanagari script check, deduplication, and a ban on the official Samasāmayik test. IN22 and FLORES are included because their licence allows training. The tokenizer is a 12,000-piece Unigram. The model is a dense 10,942,720-parameter decoder with tied embeddings, grouped-query attention, SwiGLU, RMSNorm, rotary positions, and a KV cache of 2 key-value heads.

Stage 1, base pretraining, has reached 100 million tokens. Stage 2, supervised fine-tuning on the parallel training split, is the run that teaches translation. Stage 3, DPO on preference pairs, is specified and not built. Tests cover cleaning, the tokenizer, the parameter count, the loss mask, and checkpoint resume. The pretrained model can continue text. It cannot yet translate, and it has no preference alignment.
